In [1]:
import sys
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.gridspec as grd
from accuracy import *
from data_cuts import *
import mpl_toolkits
import mpl_toolkits.axes_grid1 as axgrid
import matplotlib as mpl
import scipy.stats as stats
import os
os.environ['PATH'] = os.environ['PATH'] + ':/home/kaad8904/texlive/2024/bin/x86_64-linux'
mpl.rcParams['text.usetex'] = True  # This forces use of system LaTeX
mpl.rcParams['font.family'] = 'serif'  # Use LaTeX serif fonts
mpl.rcParams['text.latex.preamble'] = r'\usepackage{amsmath}'

In [2]:
import math

def fmt5(x):
    # Handle non-finite values
    if isinstance(x, (int, float)) and not math.isfinite(x):
        return str(x)[:5].rjust(5)

    # Try from most to fewest significant digits
    for prec in range(5, 0, -1):
        s = format(x, f".{prec}g")  # switches to sci notation as needed
        # Tidy: drop '+' in exponent and any trailing decimal point
        if 'e' in s:
            mant, exp = s.split('e')
            mant = mant.rstrip('.')
            exp = exp.lstrip('+')      # "e+10" -> "e10"
            s = f"{mant}e{exp}"
        else:
            if '.' in s:
                s = s.rstrip('0').rstrip('.')  # trim trailing zeros/point

        if len(s) <= 5:
            return s.rjust(5)  # right-align to width 5

    # Couldn't fit (extreme cases like very large negative exponents)
    return "#####"

def remove90(x,y,y_err,dependent):
    x_min = np.min(x)
    x_max = np.max(x)
    low_limit = (x_max-x_min)*(-0.1)+x_min
    high_limit = (x_max-x_min)*(1.1)+x_min
    if((len(y)!=len(y[y>low_limit])) or (len(y)!=len(y[y<high_limit]))):
        comb_mask = (y>low_limit) & (y<high_limit)
        return x[comb_mask],y[comb_mask],y_err[comb_mask],dependent[comb_mask]
    else:
        return x,y,y_err,dependent
    
def removey_err(x,y,y_err,dependent):
    x_min = np.min(x)
    x_max = np.max(x)
    x_range = x_max-x_min
    accepted_err = 2*np.exp(y_err)<x_range
    if(len(y_err)!=len(y_err[accepted_err])):
        return x[accepted_err],y[accepted_err],y_err[accepted_err],dependent[accepted_err]
    else:
        return x,y,y_err,dependent

def remove_zeros(x,y,y_err,dependent):
    mask = x != 0
    return x[mask],y[mask],y_err[mask],dependent[mask]

In [3]:
Names = ["ID","M500c","f_gas_R500c","a_form","conc_200c","shape","Gamma_R500C"\
             ,"Gamma_vir","Log_T","Log_Z","OFe","NeFe","CO","axis placeholder"\
             ,"M500c_","M200c","f_gas_0.5R200c","f_gas_0.7R200c","f_gas_R200c","f_gas_2R200c"]
Dataset_overlord = pd.read_csv("/pl/active/CASA/beop5934/halos/TNG300/Params_Groups_TNG300_z=0.00.ascii_ID_fgas", delimiter = "\s+",names = Names)
DO_new = pd.DataFrame(np.repeat(Dataset_overlord.values,3,axis=0))
DO_new.columns = Dataset_overlord.columns
ids = DO_new['ID']
ids_new = []
for i in range(0,len(ids),3):
    ids_new.append(f"{int(ids[i])}_x")
    ids_new.append(f"{int(ids[i])}_y")
    ids_new.append(f"{int(ids[i])}_z")
DO_new["ID"] = ids_new    

Names_100 = ["ID","M200c","f_gas_R500c", "a_form","conc_200c","shape","Log_M_star","M500c",
                 "f_gas_0.5R200c","f_gas_0.7R200c","f_gas_R200c","f_gas_2R200c",
                 "Log_Z_50","Log_Z_100","Log_Z_200","OFe_50","OFe_100","OFe_200","f_cool_R500c"]
Dataset_overlord_100 = pd.read_csv("/pl/active/CASA/beop5934/halos/TNG100/Params_TNG100_z=0.00.multiradii.ascii_ID_fgas", delimiter = "\s+",names = Names_100)
DO_100_new = pd.DataFrame(np.repeat(Dataset_overlord_100.values,3,axis=0))
DO_100_new.columns = Dataset_overlord_100.columns
ids = DO_100_new['ID']
ids_new = []
for i in range(0,len(ids),3):
    ids_new.append(f"{int(ids[i])}_x")
    ids_new.append(f"{int(ids[i])}_y")
    ids_new.append(f"{int(ids[i])}_z")
DO_100_new["ID"] = ids_new

f = open('CNN_Light_Alpine_inputs.txt')
input_array = f.readlines()
output_array = []
n_empt_lin = 0
for i in range(len(input_array)):
    line = input_array[i].split()
    if(len(line) > 0):
        output_array.append(line)



### Table 2 TNG100 M500

In [13]:
tipe = "Loss"
legend = "Basic"
errorbar = "Y"
colorbar = "Y"
s_l_title = "N"
add_std = "N"

choice1 = ["258","M500c","HI"]
choice2 = ["255","M500c","CCD"]
choice3 = ["265","M500c","CCD HI"]
choice4 = ["257","M500c",r"$\mu$cal"]
choice5 = ["267","M500c",r"$\mu$cal H1"]
choices = [choice1,choice2,choice3,choice4,choice5]

latex = "\\begin{center}\n"
latex += "\\begin{table*}[t]\n"
latex += "\t\\centering\n"
latex += "\t\\caption{Table 2 $M_{500}$ TNG100}\n"
latex += "\t\\begin{tabular*}{15.25cm}{|m{1.4cm}|m{0.9cm}|m{0.7cm}|m{0.7cm}|m{0.7cm}|m{0.9cm}|m{0.7cm}|m{0.7cm}|m{0.7cm}|m{0.9cm}|m{0.7cm}|m{0.7cm}|m{0.7cm}|}\n"
latex += "\t\\hline\n"
latex += "\t\\textbf{Channels} & \\multicolumn{4}{c|}{\\textbf{$<10^{12.3}$}} & "
latex += "\\multicolumn{4}{c|}{\\textbf{$10^{12.3}-10^{13.0}$}} & \\multicolumn{4}{c|}{\\textbf{$>10^{13.0}$}} \\\\\n"
latex += "\t\\cline{2-13} & RMSE & $\\rho$ & $R^2$ & $\\chi^2$ & RMSE & $\\rho$ & $R^2$ & $\\chi^2$ & RMSE & $\\rho$ & $R^2$ & $\\chi^2$ \\\\\n"
latex += "\t\\hline\n\t"
for i,choice in enumerate(choices):
    index = -1
    channels = []
    trial_number = choice[0]
    params = []
    label = choice[2]
    for j in range(len(output_array)):
        if(int(output_array[j][0]) == int(trial_number)):
            index = j

    if(index == -1):
        sys.exit('Trial Number not found, try again')
    else:
        date = output_array[index][1]
        num_channels = int(output_array[index][2])
        sim = output_array[index][3]
        print(sim)
        for j in range(num_channels):
            channels.append(output_array[index][4+j])
        num_params = int(output_array[index][4+num_channels])
        for j in range(num_params):
            params.append(output_array[index][4+num_channels+1+j])
    data_dir = "/projects/kaad8904/CNN_Light/results/"
    results_dir = data_dir+"figures/"
    df = pd.read_csv(data_dir+"CNN_Light_Trial_"+trial_number+"_data_"+tipe+".csv")
    IDs = df["ID"]
    if(sim == "100"):
        Masses = np.array([DO_100_new["M500c"][DO_100_new.index[DO_100_new["ID"]==j]] for j in IDs]).flatten()
    elif(sim == "300"):
        Masses = np.array([DO_new["M500c"][DO_new.index[DO_new["ID"]==j]] for j in IDs]).flatten()
    param_index = params.index(choice[1])
    x = df[df.keys()[3+param_index*3]].values
    y = df[df.keys()[1+param_index*3]].values
    y_err = df[df.keys()[2+param_index*3]].values
    x,y,y_err,Masses = remove90(x,y,y_err,Masses)
    x,y,y_err,Masses = removey_err(x,y,y_err,Masses)
    x,y,y_err,Masses = remove_zeros(x,y,y_err,Masses)
    if(sim == "300"):
        mask1 = Masses < 13.35
        mask2 = Masses < 14.0
        labels = ["<13.35 bin","13.35 - 14.0 bin",">14.0 bin"]
    elif(sim == "100"):
        mask1 = Masses < 12.3
        mask2 = Masses < 13.0
        labels = ["<12.3 bin","12.3 - 13.0 bin",">13.0 bin"]
    x1 = x[mask1]
    y1 = y[mask1]
    y1_err = y_err[mask1]
    x2 = x[~mask1*mask2]
    y2 = y[~mask1*mask2]
    y2_err = y_err[~mask1*mask2]
    x3 = x[~mask2]
    y3 = y[~mask2]
    y3_err = y_err[~mask2]
    RMSE = [np.nanmean(rmse(x1,y1)),np.nanmean(rmse(x2,y2)),np.nanmean(rmse(x3,y3))]
    R2 = [np.nanmean(r2_score(x1,y1)),np.nanmean(r2_score(x2,y2)),np.nanmean(r2_score(x3,y3))]
    RM_err = [np.nanmean(rel_mean_err(x1,y1)),np.nanmean(rel_mean_err(x2,y2)),np.nanmean(rel_mean_err(x3,y3))]
    chi2 = [np.nanmean(chi_squared(x1,y1,np.exp(y1_err))),np.nanmean(chi_squared(x2,y2,np.exp(y2_err))),np.nanmean(chi_squared(x3,y3,np.exp(y3_err)))]
    cor_cof = [stats.pearsonr(x1,y1)[0],stats.pearsonr(x2,y2)[0],stats.pearsonr(x3,y3)[0]]
    latex += "\\textbf{"+label+"} & " + " & ".join(map(lambda r, chi, rmse, cor_cf: f"{fmt5(rmse)} & {fmt5(cor_cf)} & {fmt5(r)} & {fmt5(chi)}",R2,chi2,RMSE,cor_cof))+" \\\\\n\t"
latex += "\\hline\n"
latex += "\t\\end{tabular*}\n"
latex += "\\label{tab:TNG"+sim+"_M500}\n"
latex += "\\end{table*}\n"
latex += "\\end{center}"
# latex += "\\hline\n\t\\textbf{HI} & "
# for row in rows:
#     latex += " & ".join(map(str, row)) + " \\\\\n"
# latex += "\\hline\n\\end{tabular}"

print(latex)

100
100
100
100
100
\begin{center}
\begin{table*}[t]
	\centering
	\caption{Table 2 $M_{500}$ TNG100}
	\begin{tabular*}{15.25cm}{|m{1.4cm}|m{0.9cm}|m{0.7cm}|m{0.7cm}|m{0.7cm}|m{0.9cm}|m{0.7cm}|m{0.7cm}|m{0.7cm}|m{0.9cm}|m{0.7cm}|m{0.7cm}|m{0.7cm}|}
	\hline
	\textbf{Channels} & \multicolumn{4}{c|}{\textbf{$<10^{12.3}$}} & \multicolumn{4}{c|}{\textbf{$10^{12.3}-10^{13.0}$}} & \multicolumn{4}{c|}{\textbf{$>10^{13.0}$}} \\
	\cline{2-13} & RMSE & $\rho$ & $R^2$ & $\chi^2$ & RMSE & $\rho$ & $R^2$ & $\chi^2$ & RMSE & $\rho$ & $R^2$ & $\chi^2$ \\
	\hline
	\textbf{HI} & 0.147 & 0.693 & 0.369 & 0.644 & 0.209 & 0.679 &  -0.1 & 0.657 & 0.536 &  -0.3 & -4.34 & 1.236 \\
	\textbf{CCD} & 0.094 & 0.877 &  0.76 & 0.806 & 0.111 &  0.92 & 0.678 & 0.656 & 0.085 & 0.966 & 0.855 & 0.298 \\
	\textbf{CCD HI} & 0.094 & 0.877 & 0.751 & 0.599 & 0.116 & 0.908 &  0.66 & 0.799 & 0.153 & 0.836 & 0.521 & 0.651 \\
	\textbf{$\mu$cal} & 0.101 & 0.851 & 0.714 & 1.025 & 0.108 & 0.878 & 0.663 & 0.658 & 0.161 & 0.804 &  0.09 

### Table 3 TNG100 f_gas_R500

In [14]:
tipe = "Loss"
legend = "Basic"
errorbar = "Y"
colorbar = "Y"
s_l_title = "N"
add_std = "N"

choice1 = ["258","f_gas_R500c","HI"]
choice2 = ["255","f_gas_R500c","CCD"]
choice3 = ["265","f_gas_R500c","CCD HI"]
choice4 = ["257","f_gas_R500c",r"$\mu$cal"]
choice5 = ["267","f_gas_R500c",r"$\mu$cal H1"]
choices = [choice1,choice2,choice3,choice4,choice5]

latex = "\\begin{center}\n"
latex += "\\begin{table*}[t]\n"
latex += "\t\\centering\n"
latex += "\t\\caption{Table 3 $f_{gas,R_{500}}$ TNG100}\n"
latex += "\t\\begin{tabular*}{15.25cm}{|m{1.4cm}|m{0.9cm}|m{0.7cm}|m{0.7cm}|m{0.7cm}|m{0.9cm}|m{0.7cm}|m{0.7cm}|m{0.7cm}|m{0.9cm}|m{0.7cm}|m{0.7cm}|m{0.7cm}|}\n"
latex += "\t\\hline\n"
latex += "\t\\textbf{Channels} & \\multicolumn{4}{c|}{\\textbf{$<10^{12.3}$}} & "
latex += "\\multicolumn{4}{c|}{\\textbf{$10^{12.3}-10^{13.0}$}} & \\multicolumn{4}{c|}{\\textbf{$>10^{13.0}$}} \\\\\n"
latex += "\t\\cline{2-13} & RMSE & $\\rho$ & $R^2$ & $\\chi^2$ & RMSE & $\\rho$ & $R^2$ & $\\chi^2$ & RMSE & $\\rho$ & $R^2$ & $\\chi^2$ \\\\\n"
latex += "\t\\hline\n\t"
for i,choice in enumerate(choices):
    index = -1
    channels = []
    trial_number = choice[0]
    params = []
    label = choice[2]
    for j in range(len(output_array)):
        if(int(output_array[j][0]) == int(trial_number)):
            index = j

    if(index == -1):
        sys.exit('Trial Number not found, try again')
    else:
        date = output_array[index][1]
        num_channels = int(output_array[index][2])
        sim = output_array[index][3]
        print(sim)
        for j in range(num_channels):
            channels.append(output_array[index][4+j])
        num_params = int(output_array[index][4+num_channels])
        for j in range(num_params):
            params.append(output_array[index][4+num_channels+1+j])
    data_dir = "/projects/kaad8904/CNN_Light/results/"
    results_dir = data_dir+"figures/"
    df = pd.read_csv(data_dir+"CNN_Light_Trial_"+trial_number+"_data_"+tipe+".csv")
    IDs = df["ID"]
    if(sim == "100"):
        Masses = np.array([DO_100_new["M500c"][DO_100_new.index[DO_100_new["ID"]==j]] for j in IDs]).flatten()
    elif(sim == "300"):
        Masses = np.array([DO_new["M500c"][DO_new.index[DO_new["ID"]==j]] for j in IDs]).flatten()
    param_index = params.index(choice[1])
    x = df[df.keys()[3+param_index*3]].values
    y = df[df.keys()[1+param_index*3]].values
    y_err = df[df.keys()[2+param_index*3]].values
    x,y,y_err,Masses = remove90(x,y,y_err,Masses)
    x,y,y_err,Masses = removey_err(x,y,y_err,Masses)
    x,y,y_err,Masses = remove_zeros(x,y,y_err,Masses)
    if(sim == "300"):
        mask1 = Masses < 13.35
        mask2 = Masses < 14.0
        labels = ["<13.35 bin","13.35 - 14.0 bin",">14.0 bin"]
    elif(sim == "100"):
        mask1 = Masses < 12.3
        mask2 = Masses < 13.0
        labels = ["<12.3 bin","12.3 - 13.0 bin",">13.0 bin"]
    x1 = x[mask1]
    y1 = y[mask1]
    y1_err = y_err[mask1]
    x2 = x[~mask1*mask2]
    y2 = y[~mask1*mask2]
    y2_err = y_err[~mask1*mask2]
    x3 = x[~mask2]
    y3 = y[~mask2]
    y3_err = y_err[~mask2]
    RMSE = [np.nanmean(rmse(x1,y1)),np.nanmean(rmse(x2,y2)),np.nanmean(rmse(x3,y3))]
    R2 = [np.nanmean(r2_score(x1,y1)),np.nanmean(r2_score(x2,y2)),np.nanmean(r2_score(x3,y3))]
    RM_err = [np.nanmean(rel_mean_err(x1,y1)),np.nanmean(rel_mean_err(x2,y2)),np.nanmean(rel_mean_err(x3,y3))]
    chi2 = [np.nanmean(chi_squared(x1,y1,np.exp(y1_err))),np.nanmean(chi_squared(x2,y2,np.exp(y2_err))),np.nanmean(chi_squared(x3,y3,np.exp(y3_err)))]
    cor_cof = [stats.pearsonr(x1,y1)[0],stats.pearsonr(x2,y2)[0],stats.pearsonr(x3,y3)[0]]
    latex += "\\textbf{"+label+"} & " + " & ".join(map(lambda r, chi, rmse, cor_cf: f"{fmt5(rmse)} & {fmt5(cor_cf)} & {fmt5(r)} & {fmt5(chi)}",R2,chi2,RMSE,cor_cof))+" \\\\\n\t"
latex += "\\hline\n"
latex += "\t\\end{tabular*}\n"
latex += "\\label{tab:TNG"+sim+"_f_gas_R500}\n"
latex += "\\end{table*}\n"
latex += "\\end{center}"
# latex += "\\hline\n\t\\textbf{HI} & "
# for row in rows:
#     latex += " & ".join(map(str, row)) + " \\\\\n"
# latex += "\\hline\n\\end{tabular}"

print(latex)

100
100
100
100
100
\begin{center}
\begin{table*}[t]
	\centering
	\caption{Table 3 $f_{gas,R_{500}}$ TNG100}
	\begin{tabular*}{15.25cm}{|m{1.4cm}|m{0.9cm}|m{0.7cm}|m{0.7cm}|m{0.7cm}|m{0.9cm}|m{0.7cm}|m{0.7cm}|m{0.7cm}|m{0.9cm}|m{0.7cm}|m{0.7cm}|m{0.7cm}|}
	\hline
	\textbf{Channels} & \multicolumn{4}{c|}{\textbf{$<10^{12.3}$}} & \multicolumn{4}{c|}{\textbf{$10^{12.3}-10^{13.0}$}} & \multicolumn{4}{c|}{\textbf{$>10^{13.0}$}} \\
	\cline{2-13} & RMSE & $\rho$ & $R^2$ & $\chi^2$ & RMSE & $\rho$ & $R^2$ & $\chi^2$ & RMSE & $\rho$ & $R^2$ & $\chi^2$ \\
	\hline
	\textbf{HI} & 0.017 & 0.893 & 0.792 & 0.744 & 0.015 & 0.727 & 0.486 & 0.629 & 0.024 & 0.299 & -0.41 & 0.716 \\
	\textbf{CCD} & 0.016 & 0.908 & 0.808 & 1.058 & 0.007 & 0.912 &  0.82 & 0.632 & 0.007 & 0.941 &  0.84 & 0.506 \\
	\textbf{CCD HI} & 0.012 & 0.947 & 0.893 & 0.833 & 0.006 & 0.945 & 0.888 & 0.376 & 0.011 & 0.879 & 0.764 &  0.46 \\
	\textbf{$\mu$cal} & 0.015 & 0.922 & 0.844 & 1.051 & 0.009 &  0.91 & 0.747 & 1.188 & 0.009 & 0.884 

### Table 4 TNG100 f_cool_R500

In [15]:
tipe = "Loss"
legend = "Basic"
errorbar = "Y"
colorbar = "Y"
s_l_title = "N"
add_std = "N"

choice1 = ["258","f_cool_R500c","HI"]
choice2 = ["255","f_cool_R500c","CCD"]
choice3 = ["265","f_cool_R500c","CCD HI"]
choice4 = ["257","f_cool_R500c",r"$\mu$cal"]
choice5 = ["267","f_cool_R500c",r"$\mu$cal H1"]
choices = [choice1,choice2,choice3,choice4,choice5]

latex = "\\begin{center}\n"
latex += "\\begin{table*}[t]\n"
latex += "\t\\centering\n"
latex += "\t\\caption{Table 4 $f_{cool,R_{500}}$ TNG100}\n"
latex += "\t\\begin{tabular*}{15.25cm}{|m{1.4cm}|m{0.9cm}|m{0.7cm}|m{0.7cm}|m{0.7cm}|m{0.9cm}|m{0.7cm}|m{0.7cm}|m{0.7cm}|m{0.9cm}|m{0.7cm}|m{0.7cm}|m{0.7cm}|}\n"
latex += "\t\\hline\n"
latex += "\t\\textbf{Channels} & \\multicolumn{4}{c|}{\\textbf{$<10^{12.3}$}} & "
latex += "\\multicolumn{4}{c|}{\\textbf{$10^{12.3}-10^{13.0}$}} & \\multicolumn{4}{c|}{\\textbf{$>10^{13.0}$}} \\\\\n"
latex += "\t\\cline{2-13} & RMSE & $\\rho$ & $R^2$ & $\\chi^2$ & RMSE & $\\rho$ & $R^2$ & $\\chi^2$ & RMSE & $\\rho$ & $R^2$ & $\\chi^2$ \\\\\n"
latex += "\t\\hline\n\t"
for i,choice in enumerate(choices):
    index = -1
    channels = []
    trial_number = choice[0]
    params = []
    label = choice[2]
    for j in range(len(output_array)):
        if(int(output_array[j][0]) == int(trial_number)):
            index = j

    if(index == -1):
        sys.exit('Trial Number not found, try again')
    else:
        date = output_array[index][1]
        num_channels = int(output_array[index][2])
        sim = output_array[index][3]
        print(sim)
        for j in range(num_channels):
            channels.append(output_array[index][4+j])
        num_params = int(output_array[index][4+num_channels])
        for j in range(num_params):
            params.append(output_array[index][4+num_channels+1+j])
    data_dir = "/projects/kaad8904/CNN_Light/results/"
    results_dir = data_dir+"figures/"
    df = pd.read_csv(data_dir+"CNN_Light_Trial_"+trial_number+"_data_"+tipe+".csv")
    IDs = df["ID"]
    if(sim == "100"):
        Masses = np.array([DO_100_new["M500c"][DO_100_new.index[DO_100_new["ID"]==j]] for j in IDs]).flatten()
    elif(sim == "300"):
        Masses = np.array([DO_new["M500c"][DO_new.index[DO_new["ID"]==j]] for j in IDs]).flatten()
    param_index = params.index(choice[1])
    x = df[df.keys()[3+param_index*3]].values
    y = df[df.keys()[1+param_index*3]].values
    y_err = df[df.keys()[2+param_index*3]].values
    x,y,y_err,Masses = remove90(x,y,y_err,Masses)
    x,y,y_err,Masses = removey_err(x,y,y_err,Masses)
    x,y,y_err,Masses = remove_zeros(x,y,y_err,Masses)
    if(sim == "300"):
        mask1 = Masses < 13.35
        mask2 = Masses < 14.0
        labels = ["<13.35 bin","13.35 - 14.0 bin",">14.0 bin"]
    elif(sim == "100"):
        mask1 = Masses < 12.3
        mask2 = Masses < 13.0
        labels = ["<12.3 bin","12.3 - 13.0 bin",">13.0 bin"]
    x1 = x[mask1]
    y1 = y[mask1]
    y1_err = y_err[mask1]
    x2 = x[~mask1*mask2]
    y2 = y[~mask1*mask2]
    y2_err = y_err[~mask1*mask2]
    x3 = x[~mask2]
    y3 = y[~mask2]
    y3_err = y_err[~mask2]
    RMSE = [np.nanmean(rmse(x1,y1)),np.nanmean(rmse(x2,y2)),np.nanmean(rmse(x3,y3))]
    R2 = [np.nanmean(r2_score(x1,y1)),np.nanmean(r2_score(x2,y2)),np.nanmean(r2_score(x3,y3))]
    RM_err = [np.nanmean(rel_mean_err(x1,y1)),np.nanmean(rel_mean_err(x2,y2)),np.nanmean(rel_mean_err(x3,y3))]
    chi2 = [np.nanmean(chi_squared(x1,y1,np.exp(y1_err))),np.nanmean(chi_squared(x2,y2,np.exp(y2_err))),np.nanmean(chi_squared(x3,y3,np.exp(y3_err)))]
    cor_cof = [stats.pearsonr(x1,y1)[0],stats.pearsonr(x2,y2)[0],stats.pearsonr(x3,y3)[0]]
    latex += "\\textbf{"+label+"} & " + " & ".join(map(lambda r, chi, rmse, cor_cf: f"{fmt5(rmse)} & {fmt5(cor_cf)} & {fmt5(r)} & {fmt5(chi)}",R2,chi2,RMSE,cor_cof))+" \\\\\n\t"
latex += "\\hline\n"
latex += "\t\\end{tabular*}\n"
latex += "\\label{tab:TNG"+sim+"_f_cool_R500}\n"
latex += "\\end{table*}\n"
latex += "\\end{center}"
# latex += "\\hline\n\t\\textbf{HI} & "
# for row in rows:
#     latex += " & ".join(map(str, row)) + " \\\\\n"
# latex += "\\hline\n\\end{tabular}"

print(latex)

100
100
100
100
100
\begin{center}
\begin{table*}[t]
	\centering
	\caption{Table 4 $f_{cool,R_{500}}$ TNG100}
	\begin{tabular*}{15.25cm}{|m{1.4cm}|m{0.9cm}|m{0.7cm}|m{0.7cm}|m{0.7cm}|m{0.9cm}|m{0.7cm}|m{0.7cm}|m{0.7cm}|m{0.9cm}|m{0.7cm}|m{0.7cm}|m{0.7cm}|}
	\hline
	\textbf{Channels} & \multicolumn{4}{c|}{\textbf{$<10^{12.3}$}} & \multicolumn{4}{c|}{\textbf{$10^{12.3}-10^{13.0}$}} & \multicolumn{4}{c|}{\textbf{$>10^{13.0}$}} \\
	\cline{2-13} & RMSE & $\rho$ & $R^2$ & $\chi^2$ & RMSE & $\rho$ & $R^2$ & $\chi^2$ & RMSE & $\rho$ & $R^2$ & $\chi^2$ \\
	\hline
	\textbf{HI} & 0.096 & 0.882 & 0.771 & 0.748 & 0.114 &  0.86 & 0.697 & 0.789 & 0.106 & 0.491 &  -1.5 & 0.376 \\
	\textbf{CCD} &  0.12 & 0.795 & 0.625 & 0.909 & 0.114 & 0.795 & 0.607 & 0.881 & 0.048 & 0.873 & 0.599 & 0.338 \\
	\textbf{CCD HI} & 0.069 & 0.939 & 0.879 & 0.811 & 0.058 & 0.956 & 0.909 & 0.628 & 0.042 & 0.728 & 0.406 & 0.347 \\
	\textbf{$\mu$cal} & 0.108 & 0.842 & 0.707 & 0.949 & 0.107 & 0.861 & 0.711 & 1.046 & 0.045 & 0.615

### Table 7 TNG 300 M500

In [22]:
choice1 = ["254","M500c","CCD"]
choice2 = ["264","M500c","CCD HI"]
choice3 = ["260","M500c","CCD HI,vel, disp"]
choices = [choice1,choice2,choice3]

latex = "\\begin{center}\n"
latex += "\\begin{table*}[t]\n"
latex += "\t\\centering\n"
latex += "\t\\caption{Table 7 $M_{500}$ TNG300}\n"
latex += "\t\\begin{tabular*}{15.25cm}{|m{1.4cm}|m{0.9cm}|m{0.7cm}|m{0.7cm}|m{0.7cm}|m{0.9cm}|m{0.7cm}|m{0.7cm}|m{0.7cm}|m{0.9cm}|m{0.7cm}|m{0.7cm}|m{0.7cm}|}\n"
latex += "\t\\hline\n"
latex += "\t\\textbf{Channels} & \\multicolumn{4}{c|}{\\textbf{$<10^{13.35}$}} & "
latex += "\\multicolumn{4}{c|}{\\textbf{$10^{13.35}-10^{14.0}$}} & \\multicolumn{4}{c|}{\\textbf{$>10^{14.0}$}} \\\\\n"
latex += "\t\\cline{2-13} & RMSE & $\\rho$ & $R^2$ & $\\chi^2$ & RMSE & $\\rho$ & $R^2$ & $\\chi^2$ & RMSE & $\\rho$ & $R^2$ & $\\chi^2$ \\\\\n"
latex += "\t\\hline\n\t"
for i,choice in enumerate(choices):
    index = -1
    channels = []
    trial_number = choice[0]
    params = []
    label = choice[2]
    for j in range(len(output_array)):
        if(int(output_array[j][0]) == int(trial_number)):
            index = j

    if(index == -1):
        sys.exit('Trial Number not found, try again')
    else:
        date = output_array[index][1]
        num_channels = int(output_array[index][2])
        sim = output_array[index][3]
        print(sim)
        for j in range(num_channels):
            channels.append(output_array[index][4+j])
        num_params = int(output_array[index][4+num_channels])
        for j in range(num_params):
            params.append(output_array[index][4+num_channels+1+j])
    data_dir = "/projects/kaad8904/CNN_Light/results/"
    results_dir = data_dir+"figures/"
    df = pd.read_csv(data_dir+"CNN_Light_Trial_"+trial_number+"_data_"+tipe+".csv")
    IDs = df["ID"]
    if(sim == "100"):
        Masses = np.array([DO_100_new["M500c"][DO_100_new.index[DO_100_new["ID"]==j]] for j in IDs]).flatten()
    elif(sim == "300"):
        Masses = np.array([DO_new["M500c"][DO_new.index[DO_new["ID"]==j]] for j in IDs]).flatten()
    param_index = params.index(choice[1])
    x = df[df.keys()[3+param_index*3]].values
    y = df[df.keys()[1+param_index*3]].values
    y_err = df[df.keys()[2+param_index*3]].values
    x,y,y_err,Masses = remove90(x,y,y_err,Masses)
    x,y,y_err,Masses = removey_err(x,y,y_err,Masses)
    x,y,y_err,Masses = remove_zeros(x,y,y_err,Masses)
    if(sim == "300"):
        mask1 = Masses < 13.35
        mask2 = Masses < 14.0
        labels = ["<13.35 bin","13.35 - 14.0 bin",">14.0 bin"]
    elif(sim == "100"):
        mask1 = Masses < 12.3
        mask2 = Masses < 13.0
        labels = ["<12.3 bin","12.3 - 13.0 bin",">13.0 bin"]
    x1 = x[mask1]
    y1 = y[mask1]
    y1_err = y_err[mask1]
    x2 = x[~mask1*mask2]
    y2 = y[~mask1*mask2]
    y2_err = y_err[~mask1*mask2]
    x3 = x[~mask2]
    y3 = y[~mask2]
    y3_err = y_err[~mask2]
    RMSE = [np.nanmean(rmse(x1,y1)),np.nanmean(rmse(x2,y2)),np.nanmean(rmse(x3,y3))]
    R2 = [np.nanmean(r2_score(x1,y1)),np.nanmean(r2_score(x2,y2)),np.nanmean(r2_score(x3,y3))]
    RM_err = [np.nanmean(rel_mean_err(x1,y1)),np.nanmean(rel_mean_err(x2,y2)),np.nanmean(rel_mean_err(x3,y3))]
    chi2 = [np.nanmean(chi_squared(x1,y1,np.exp(y1_err))),np.nanmean(chi_squared(x2,y2,np.exp(y2_err))),np.nanmean(chi_squared(x3,y3,np.exp(y3_err)))]
    cor_cof = [stats.pearsonr(x1,y1)[0],stats.pearsonr(x2,y2)[0],stats.pearsonr(x3,y3)[0]]
    latex += "\\textbf{"+label+"} & " + " & ".join(map(lambda r, chi, rmse, cor_cf: f"{fmt5(rmse)} & {fmt5(cor_cf)} & {fmt5(r)} & {fmt5(chi)}",R2,chi2,RMSE,cor_cof))+" \\\\\n\t"
latex += "\\hline\n"
latex += "\t\\end{tabular*}\n"
latex += "\\label{tab:TNG"+sim+"_M500}\n"
latex += "\\end{table*}\n"
latex += "\\end{center}"
# latex += "\\hline\n\t\\textbf{HI} & "
# for row in rows:
#     latex += " & ".join(map(str, row)) + " \\\\\n"
# latex += "\\hline\n\\end{tabular}"

print(latex)

300
300
300
\begin{center}
\begin{table*}[t]
	\centering
	\caption{Table 7 $M_{500}$ TNG300}
	\begin{tabular*}{15.25cm}{|m{1.4cm}|m{0.9cm}|m{0.7cm}|m{0.7cm}|m{0.7cm}|m{0.9cm}|m{0.7cm}|m{0.7cm}|m{0.7cm}|m{0.9cm}|m{0.7cm}|m{0.7cm}|m{0.7cm}|}
	\hline
	\textbf{Channels} & \multicolumn{4}{c|}{\textbf{$<10^{13.35}$}} & \multicolumn{4}{c|}{\textbf{$10^{13.35}-10^{14.0}$}} & \multicolumn{4}{c|}{\textbf{$>10^{14.0}$}} \\
	\cline{2-13} & RMSE & $\rho$ & $R^2$ & $\chi^2$ & RMSE & $\rho$ & $R^2$ & $\chi^2$ & RMSE & $\rho$ & $R^2$ & $\chi^2$ \\
	\hline
	\textbf{CCD} & 0.041 & 0.958 & 0.911 & 0.708 & 0.046 & 0.982 &  0.93 &  0.82 & 0.156 & -0.25 & -2.01 & 1.176 \\
	\textbf{CCD HI} & 0.045 & 0.945 &  0.89 & 0.704 & 0.033 & 0.981 & 0.961 & 0.321 & 0.096 & 0.777 &  -0.6 & 0.282 \\
	\textbf{CCD HI,vel, disp} & 0.049 & 0.945 & 0.867 & 0.854 & 0.041 & 0.978 & 0.944 & 0.631 & 0.099 & 0.911 & 0.506 & 0.309 \\
	\hline
	\end{tabular*}
\label{tab:TNG300_M500}
\end{table*}
\end{center}


### Table 7 TNG300 f_gas_R500

In [21]:
choice1 = ["254","f_gas_R500c","CCD"]
choice2 = ["264","f_gas_R500c","CCD HI"]
choice3 = ["260","f_gas_R500c","CCD HI,vel, disp"]
choices = [choice1,choice2,choice3]

latex = "\\begin{center}\n"
latex += "\\begin{table*}[t]\n"
latex += "\t\\centering\n"
latex += "\t\\caption{Table 7 $f_{gas,R_{500}}$ TNG300}\n"
latex += "\t\\begin{tabular*}{15.25cm}{|m{1.4cm}|m{0.9cm}|m{0.7cm}|m{0.7cm}|m{0.7cm}|m{0.9cm}|m{0.7cm}|m{0.7cm}|m{0.7cm}|m{0.9cm}|m{0.7cm}|m{0.7cm}|m{0.7cm}|}\n"
latex += "\t\\hline\n"
latex += "\t\\textbf{Channels} & \\multicolumn{4}{c|}{\\textbf{$<10^{13.35}$}} & "
latex += "\\multicolumn{4}{c|}{\\textbf{$10^{13.35}-10^{14.0}$}} & \\multicolumn{4}{c|}{\\textbf{$>10^{14.0}$}} \\\\\n"
latex += "\t\\cline{2-13} & RMSE & $\\rho$ & $R^2$ & $\\chi^2$ & RMSE & $\\rho$ & $R^2$ & $\\chi^2$ & RMSE & $\\rho$ & $R^2$ & $\\chi^2$ \\\\\n"
latex += "\t\\hline\n\t"
for i,choice in enumerate(choices):
    index = -1
    channels = []
    trial_number = choice[0]
    params = []
    label = choice[2]
    for j in range(len(output_array)):
        if(int(output_array[j][0]) == int(trial_number)):
            index = j

    if(index == -1):
        sys.exit('Trial Number not found, try again')
    else:
        date = output_array[index][1]
        num_channels = int(output_array[index][2])
        sim = output_array[index][3]
        print(sim)
        for j in range(num_channels):
            channels.append(output_array[index][4+j])
        num_params = int(output_array[index][4+num_channels])
        for j in range(num_params):
            params.append(output_array[index][4+num_channels+1+j])
    data_dir = "/projects/kaad8904/CNN_Light/results/"
    results_dir = data_dir+"figures/"
    df = pd.read_csv(data_dir+"CNN_Light_Trial_"+trial_number+"_data_"+tipe+".csv")
    IDs = df["ID"]
    if(sim == "100"):
        Masses = np.array([DO_100_new["M500c"][DO_100_new.index[DO_100_new["ID"]==j]] for j in IDs]).flatten()
    elif(sim == "300"):
        Masses = np.array([DO_new["M500c"][DO_new.index[DO_new["ID"]==j]] for j in IDs]).flatten()
    param_index = params.index(choice[1])
    x = df[df.keys()[3+param_index*3]].values
    y = df[df.keys()[1+param_index*3]].values
    y_err = df[df.keys()[2+param_index*3]].values
    x,y,y_err,Masses = remove90(x,y,y_err,Masses)
    x,y,y_err,Masses = removey_err(x,y,y_err,Masses)
    x,y,y_err,Masses = remove_zeros(x,y,y_err,Masses)
    if(sim == "300"):
        mask1 = Masses < 13.35
        mask2 = Masses < 14.0
        labels = ["<13.35 bin","13.35 - 14.0 bin",">14.0 bin"]
    elif(sim == "100"):
        mask1 = Masses < 12.3
        mask2 = Masses < 13.0
        labels = ["<12.3 bin","12.3 - 13.0 bin",">13.0 bin"]
    x1 = x[mask1]
    y1 = y[mask1]
    y1_err = y_err[mask1]
    x2 = x[~mask1*mask2]
    y2 = y[~mask1*mask2]
    y2_err = y_err[~mask1*mask2]
    x3 = x[~mask2]
    y3 = y[~mask2]
    y3_err = y_err[~mask2]
    RMSE = [np.nanmean(rmse(x1,y1)),np.nanmean(rmse(x2,y2)),np.nanmean(rmse(x3,y3))]
    R2 = [np.nanmean(r2_score(x1,y1)),np.nanmean(r2_score(x2,y2)),np.nanmean(r2_score(x3,y3))]
    RM_err = [np.nanmean(rel_mean_err(x1,y1)),np.nanmean(rel_mean_err(x2,y2)),np.nanmean(rel_mean_err(x3,y3))]
    chi2 = [np.nanmean(chi_squared(x1,y1,np.exp(y1_err))),np.nanmean(chi_squared(x2,y2,np.exp(y2_err))),np.nanmean(chi_squared(x3,y3,np.exp(y3_err)))]
    cor_cof = [stats.pearsonr(x1,y1)[0],stats.pearsonr(x2,y2)[0],stats.pearsonr(x3,y3)[0]]
    latex += "\\textbf{"+label+"} & " + " & ".join(map(lambda r, chi, rmse, cor_cf: f"{fmt5(rmse)} & {fmt5(cor_cf)} & {fmt5(r)} & {fmt5(chi)}",R2,chi2,RMSE,cor_cof))+" \\\\\n\t"
latex += "\\hline\n"
latex += "\t\\end{tabular*}\n"
latex += "\\label{tab:TNG"+sim+"_f_gas_R500}\n"
latex += "\\end{table*}\n"
latex += "\\end{center}"
# latex += "\\hline\n\t\\textbf{HI} & "
# for row in rows:
#     latex += " & ".join(map(str, row)) + " \\\\\n"
# latex += "\\hline\n\\end{tabular}"

print(latex)

300
300
300
\begin{center}
\begin{table*}[t]
	\centering
	\caption{Table 7 $f_{gas,R_{500}}$ TNG300}
	\begin{tabular*}{15.25cm}{|m{1.4cm}|m{0.9cm}|m{0.7cm}|m{0.7cm}|m{0.7cm}|m{0.9cm}|m{0.7cm}|m{0.7cm}|m{0.7cm}|m{0.9cm}|m{0.7cm}|m{0.7cm}|m{0.7cm}|}
	\hline
	\textbf{Channels} & \multicolumn{4}{c|}{\textbf{$<10^{13.35}$}} & \multicolumn{4}{c|}{\textbf{$10^{13.35}-10^{14.0}$}} & \multicolumn{4}{c|}{\textbf{$>10^{14.0}$}} \\
	\cline{2-13} & RMSE & $\rho$ & $R^2$ & $\chi^2$ & RMSE & $\rho$ & $R^2$ & $\chi^2$ & RMSE & $\rho$ & $R^2$ & $\chi^2$ \\
	\hline
	\textbf{CCD} & 0.004 & 0.958 & 0.893 & 1.051 & 0.005 & 0.958 & 0.916 & 0.881 & 0.009 & 0.252 &  -0.2 & 0.525 \\
	\textbf{CCD HI} & 0.004 & 0.966 & 0.916 & 0.718 & 0.006 & 0.961 & 0.891 &  0.85 & 0.012 & 0.282 & -1.22 & 0.229 \\
	\textbf{CCD HI,vel, disp} & 0.004 & 0.967 & 0.929 & 0.735 & 0.005 & 0.962 & 0.919 & 0.944 & 0.009 & 0.674 & 0.393 & 0.481 \\
	\hline
	\end{tabular*}
\label{tab:TNG300_f_gas_R500}
\end{table*}
\end{center}


### Table 6 TNG100 Metallicity log_Z

In [16]:
choice1 = ["268","Log_Z_100","CCD"]
choice2 = ["269","Log_Z_100",r"$\mu$cal"]
choices = [choice1,choice2]

latex = "\\begin{center}\n"
latex += "\\begin{table*}[t]\n"
latex += "\t\\centering\n"
latex += "\t\\caption{Table 6 \\Log{Z} TNG100}\n"
latex += "\t\\begin{tabular*}{15.25cm}{|m{1.4cm}|m{0.9cm}|m{0.7cm}|m{0.7cm}|m{0.7cm}|m{0.9cm}|m{0.7cm}|m{0.7cm}|m{0.7cm}|m{0.9cm}|m{0.7cm}|m{0.7cm}|m{0.7cm}|}\n"
latex += "\t\\hline\n"
latex += "\t\\textbf{Channels} & \\multicolumn{4}{c|}{\\textbf{$<10^{12.3}$}} & "
latex += "\\multicolumn{4}{c|}{\\textbf{$10^{12.3}-10^{13.0}$}} & \\multicolumn{4}{c|}{\\textbf{$>10^{13.0}$}} \\\\\n"
latex += "\t\\cline{2-13} & RMSE & $\\rho$ & $R^2$ & $\\chi^2$ & RMSE & $\\rho$ & $R^2$ & $\\chi^2$ & RMSE & $\\rho$ & $R^2$ & $\\chi^2$ \\\\\n"
latex += "\t\\hline\n\t"
for i,choice in enumerate(choices):
    index = -1
    channels = []
    trial_number = choice[0]
    params = []
    label = choice[2]
    for j in range(len(output_array)):
        if(int(output_array[j][0]) == int(trial_number)):
            index = j

    if(index == -1):
        sys.exit('Trial Number not found, try again')
    else:
        date = output_array[index][1]
        num_channels = int(output_array[index][2])
        sim = output_array[index][3]
        print(sim)
        for j in range(num_channels):
            channels.append(output_array[index][4+j])
        num_params = int(output_array[index][4+num_channels])
        for j in range(num_params):
            params.append(output_array[index][4+num_channels+1+j])
    data_dir = "/projects/kaad8904/CNN_Light/results/"
    results_dir = data_dir+"figures/"
    df = pd.read_csv(data_dir+"CNN_Light_Trial_"+trial_number+"_data_"+tipe+".csv")
    IDs = df["ID"]
    if(sim == "100"):
        Masses = np.array([DO_100_new["M500c"][DO_100_new.index[DO_100_new["ID"]==j]] for j in IDs]).flatten()
    elif(sim == "300"):
        Masses = np.array([DO_new["M500c"][DO_new.index[DO_new["ID"]==j]] for j in IDs]).flatten()
    param_index = params.index(choice[1])
    x = df[df.keys()[3+param_index*3]].values
    y = df[df.keys()[1+param_index*3]].values
    y_err = df[df.keys()[2+param_index*3]].values
    x,y,y_err,Masses = remove90(x,y,y_err,Masses)
    x,y,y_err,Masses = removey_err(x,y,y_err,Masses)
    x,y,y_err,Masses = remove_zeros(x,y,y_err,Masses)
    if(sim == "300"):
        mask1 = Masses < 13.35
        mask2 = Masses < 14.0
        labels = ["<13.35 bin","13.35 - 14.0 bin",">14.0 bin"]
    elif(sim == "100"):
        mask1 = Masses < 12.3
        mask2 = Masses < 13.0
        labels = ["<12.3 bin","12.3 - 13.0 bin",">13.0 bin"]
    x1 = x[mask1]
    y1 = y[mask1]
    y1_err = y_err[mask1]
    x2 = x[~mask1*mask2]
    y2 = y[~mask1*mask2]
    y2_err = y_err[~mask1*mask2]
    x3 = x[~mask2]
    y3 = y[~mask2]
    y3_err = y_err[~mask2]
    RMSE = [np.nanmean(rmse(x1,y1)),np.nanmean(rmse(x2,y2)),np.nanmean(rmse(x3,y3))]
    R2 = [np.nanmean(r2_score(x1,y1)),np.nanmean(r2_score(x2,y2)),np.nanmean(r2_score(x3,y3))]
    RM_err = [np.nanmean(rel_mean_err(x1,y1)),np.nanmean(rel_mean_err(x2,y2)),np.nanmean(rel_mean_err(x3,y3))]
    chi2 = [np.nanmean(chi_squared(x1,y1,np.exp(y1_err))),np.nanmean(chi_squared(x2,y2,np.exp(y2_err))),np.nanmean(chi_squared(x3,y3,np.exp(y3_err)))]
    cor_cof = [stats.pearsonr(x1,y1)[0],stats.pearsonr(x2,y2)[0],stats.pearsonr(x3,y3)[0]]
    latex += "\\textbf{"+label+"} & " + " & ".join(map(lambda r, chi, rmse, cor_cf: f"{fmt5(rmse)} & {fmt5(cor_cf)} & {fmt5(r)} & {fmt5(chi)}",R2,chi2,RMSE,cor_cof))+" \\\\\n\t"
latex += "\\hline\n"
latex += "\t\\end{tabular*}\n"
latex += "\\label{tab:Log_Z_TNG"+sim+"}\n"
latex += "\\end{table*}\n"
latex += "\\end{center}"
# latex += "\\hline\n\t\\textbf{HI} & "
# for row in rows:
#     latex += " & ".join(map(str, row)) + " \\\\\n"
# latex += "\\hline\n\\end{tabular}"

print(latex)

100
100
\begin{center}
\begin{table*}[t]
	\centering
	\caption{Table 6 \Log{Z} TNG100}
	\begin{tabular*}{15.25cm}{|m{1.4cm}|m{0.9cm}|m{0.7cm}|m{0.7cm}|m{0.7cm}|m{0.9cm}|m{0.7cm}|m{0.7cm}|m{0.7cm}|m{0.9cm}|m{0.7cm}|m{0.7cm}|m{0.7cm}|}
	\hline
	\textbf{Channels} & \multicolumn{4}{c|}{\textbf{$<10^{12.3}$}} & \multicolumn{4}{c|}{\textbf{$10^{12.3}-10^{13.0}$}} & \multicolumn{4}{c|}{\textbf{$>10^{13.0}$}} \\
	\cline{2-13} & RMSE & $\rho$ & $R^2$ & $\chi^2$ & RMSE & $\rho$ & $R^2$ & $\chi^2$ & RMSE & $\rho$ & $R^2$ & $\chi^2$ \\
	\hline
	\textbf{CCD} & 0.112 & 0.656 & 0.427 & 0.748 &  0.15 & 0.678 & 0.439 & 0.948 & 0.162 & 0.627 & 0.282 & 0.867 \\
	\textbf{$\mu$cal} & 0.095 & 0.794 & 0.625 & 0.958 & 0.085 & 0.917 & 0.833 & 0.757 & 0.095 & 0.832 & 0.682 & 0.718 \\
	\hline
	\end{tabular*}
\label{tab:Log_Z_TNG100}
\end{table*}
\end{center}


### Table 6 TNG100 Metallicity OFe

In [17]:
choice1 = ["268","OFe_100","CCD"]
choice2 = ["269","OFe_100",r"$\mu$cal"]
choices = [choice1,choice2]

latex = "\\begin{center}\n"
latex += "\\begin{table*}[t]\n"
latex += "\t\\centering\n"
latex += "\t\\caption{Table 6 [O/Fe] TNG100}\n"
latex += "\t\\begin{tabular*}{15.25cm}{|m{1.4cm}|m{0.9cm}|m{0.7cm}|m{0.7cm}|m{0.7cm}|m{0.9cm}|m{0.7cm}|m{0.7cm}|m{0.7cm}|m{0.9cm}|m{0.7cm}|m{0.7cm}|m{0.7cm}|}\n"
latex += "\t\\hline\n"
latex += "\t\\textbf{Channels} & \\multicolumn{4}{c|}{\\textbf{$<10^{12.3}$}} & "
latex += "\\multicolumn{4}{c|}{\\textbf{$10^{12.3}-10^{13.0}$}} & \\multicolumn{4}{c|}{\\textbf{$>10^{13.0}$}} \\\\\n"
latex += "\t\\cline{2-13} & RMSE & $\\rho$ & $R^2$ & $\\chi^2$ & RMSE & $\\rho$ & $R^2$ & $\\chi^2$ & RMSE & $\\rho$ & $R^2$ & $\\chi^2$ \\\\\n"
latex += "\t\\hline\n\t"
for i,choice in enumerate(choices):
    index = -1
    channels = []
    trial_number = choice[0]
    params = []
    label = choice[2]
    for j in range(len(output_array)):
        if(int(output_array[j][0]) == int(trial_number)):
            index = j

    if(index == -1):
        sys.exit('Trial Number not found, try again')
    else:
        date = output_array[index][1]
        num_channels = int(output_array[index][2])
        sim = output_array[index][3]
        print(sim)
        for j in range(num_channels):
            channels.append(output_array[index][4+j])
        num_params = int(output_array[index][4+num_channels])
        for j in range(num_params):
            params.append(output_array[index][4+num_channels+1+j])
    data_dir = "/projects/kaad8904/CNN_Light/results/"
    results_dir = data_dir+"figures/"
    df = pd.read_csv(data_dir+"CNN_Light_Trial_"+trial_number+"_data_"+tipe+".csv")
    IDs = df["ID"]
    if(sim == "100"):
        Masses = np.array([DO_100_new["M500c"][DO_100_new.index[DO_100_new["ID"]==j]] for j in IDs]).flatten()
    elif(sim == "300"):
        Masses = np.array([DO_new["M500c"][DO_new.index[DO_new["ID"]==j]] for j in IDs]).flatten()
    param_index = params.index(choice[1])
    x = df[df.keys()[3+param_index*3]].values
    y = df[df.keys()[1+param_index*3]].values
    y_err = df[df.keys()[2+param_index*3]].values
    x,y,y_err,Masses = remove90(x,y,y_err,Masses)
    x,y,y_err,Masses = removey_err(x,y,y_err,Masses)
    x,y,y_err,Masses = remove_zeros(x,y,y_err,Masses)
    if(sim == "300"):
        mask1 = Masses < 13.35
        mask2 = Masses < 14.0
        labels = ["<13.35 bin","13.35 - 14.0 bin",">14.0 bin"]
    elif(sim == "100"):
        mask1 = Masses < 12.3
        mask2 = Masses < 13.0
        labels = ["<12.3 bin","12.3 - 13.0 bin",">13.0 bin"]
    x1 = x[mask1]
    y1 = y[mask1]
    y1_err = y_err[mask1]
    x2 = x[~mask1*mask2]
    y2 = y[~mask1*mask2]
    y2_err = y_err[~mask1*mask2]
    x3 = x[~mask2]
    y3 = y[~mask2]
    y3_err = y_err[~mask2]
    RMSE = [np.nanmean(rmse(x1,y1)),np.nanmean(rmse(x2,y2)),np.nanmean(rmse(x3,y3))]
    R2 = [np.nanmean(r2_score(x1,y1)),np.nanmean(r2_score(x2,y2)),np.nanmean(r2_score(x3,y3))]
    RM_err = [np.nanmean(rel_mean_err(x1,y1)),np.nanmean(rel_mean_err(x2,y2)),np.nanmean(rel_mean_err(x3,y3))]
    chi2 = [np.nanmean(chi_squared(x1,y1,np.exp(y1_err))),np.nanmean(chi_squared(x2,y2,np.exp(y2_err))),np.nanmean(chi_squared(x3,y3,np.exp(y3_err)))]
    cor_cof = [stats.pearsonr(x1,y1)[0],stats.pearsonr(x2,y2)[0],stats.pearsonr(x3,y3)[0]]
    latex += "\\textbf{"+label+"} & " + " & ".join(map(lambda r, chi, rmse, cor_cf: f"{fmt5(rmse)} & {fmt5(cor_cf)} & {fmt5(r)} & {fmt5(chi)}",R2,chi2,RMSE,cor_cof))+" \\\\\n\t"
latex += "\\hline\n"
latex += "\t\\end{tabular*}\n"
latex += "\\label{tab:OFe_TNG"+sim+"}\n"
latex += "\\end{table*}\n"
latex += "\\end{center}"
# latex += "\\hline\n\t\\textbf{HI} & "
# for row in rows:
#     latex += " & ".join(map(str, row)) + " \\\\\n"
# latex += "\\hline\n\\end{tabular}"

print(latex)

100
100
\begin{center}
\begin{table*}[t]
	\centering
	\caption{Table 6 [O/Fe] TNG100}
	\begin{tabular*}{15.25cm}{|m{1.4cm}|m{0.9cm}|m{0.7cm}|m{0.7cm}|m{0.7cm}|m{0.9cm}|m{0.7cm}|m{0.7cm}|m{0.7cm}|m{0.9cm}|m{0.7cm}|m{0.7cm}|m{0.7cm}|}
	\hline
	\textbf{Channels} & \multicolumn{4}{c|}{\textbf{$<10^{12.3}$}} & \multicolumn{4}{c|}{\textbf{$10^{12.3}-10^{13.0}$}} & \multicolumn{4}{c|}{\textbf{$>10^{13.0}$}} \\
	\cline{2-13} & RMSE & $\rho$ & $R^2$ & $\chi^2$ & RMSE & $\rho$ & $R^2$ & $\chi^2$ & RMSE & $\rho$ & $R^2$ & $\chi^2$ \\
	\hline
	\textbf{CCD} & 0.052 & 0.918 & 0.841 & 0.804 & 0.059 & 0.854 & 0.718 & 0.787 & 0.047 & 0.716 & 0.461 & 0.479 \\
	\textbf{$\mu$cal} & 0.045 & 0.949 & 0.897 & 0.812 &  0.04 & 0.945 & 0.888 & 0.737 & 0.049 & 0.841 & 0.652 & 0.971 \\
	\hline
	\end{tabular*}
\label{tab:OFe_TNG100}
\end{table*}
\end{center}


### Table 5 TNG100 H1 Vs. H1_Multi

In [18]:
choice1 = ["258","M500c","H1"]
choice2 = ["259","M500c","H1,vel,disp"]
choice3 = ["258","f_gas_R500c","H1"]
choice4 = ["259","f_gas_R500c","H1,vel,disp"]
choice5 = ["258","f_cool_R500c","H1"]
choice6 = ["259","f_cool_R500c","H1,vel,disp"]
choices = [choice1,choice2,choice3,choice4,choice5,choice6]

latex = "\\begin{center}\n"
latex += "\\begin{table*}[t]\n"
latex += "\t\\centering\n"
latex += "\t\\caption{Table 5 \HI\ vs \HI\,velocity, and dispersion}\n"
latex += "\t\\begin{tabular*}{15.25cm}{|m{1.4cm}|m{0.9cm}|m{0.7cm}|m{0.7cm}|m{0.7cm}|m{0.9cm}|m{0.7cm}|m{0.7cm}|m{0.7cm}|m{0.9cm}|m{0.7cm}|m{0.7cm}|m{0.7cm}|}\n"
latex += "\t\\hline\n"
latex += "\t\\textbf{Channels} & \\multicolumn{4}{c|}{\\textbf{$<10^{12.3}$}} & "
latex += "\\multicolumn{4}{c|}{\\textbf{$10^{12.3}-10^{13.0}$}} & \\multicolumn{4}{c|}{\\textbf{$>10^{13.0}$}} \\\\\n"
latex += "\t\\cline{2-13} & RMSE & $\\rho$ & $R^2$ & $\\chi^2$ & RMSE & $\\rho$ & $R^2$ & $\\chi^2$ & RMSE & $\\rho$ & $R^2$ & $\\chi^2$ \\\\\n"
latex += "\t\\hline\n\t"
for i,choice in enumerate(choices):
    index = -1
    channels = []
    trial_number = choice[0]
    params = []
    label = choice[2]
    for j in range(len(output_array)):
        if(int(output_array[j][0]) == int(trial_number)):
            index = j

    if(index == -1):
        sys.exit('Trial Number not found, try again')
    else:
        date = output_array[index][1]
        num_channels = int(output_array[index][2])
        sim = output_array[index][3]
        print(sim)
        for j in range(num_channels):
            channels.append(output_array[index][4+j])
        num_params = int(output_array[index][4+num_channels])
        for j in range(num_params):
            params.append(output_array[index][4+num_channels+1+j])
    data_dir = "/projects/kaad8904/CNN_Light/results/"
    results_dir = data_dir+"figures/"
    df = pd.read_csv(data_dir+"CNN_Light_Trial_"+trial_number+"_data_"+tipe+".csv")
    IDs = df["ID"]
    if(sim == "100"):
        Masses = np.array([DO_100_new["M500c"][DO_100_new.index[DO_100_new["ID"]==j]] for j in IDs]).flatten()
    elif(sim == "300"):
        Masses = np.array([DO_new["M500c"][DO_new.index[DO_new["ID"]==j]] for j in IDs]).flatten()
    param_index = params.index(choice[1])
    x = df[df.keys()[3+param_index*3]].values
    y = df[df.keys()[1+param_index*3]].values
    y_err = df[df.keys()[2+param_index*3]].values
    x,y,y_err,Masses = remove90(x,y,y_err,Masses)
    x,y,y_err,Masses = removey_err(x,y,y_err,Masses)
    x,y,y_err,Masses = remove_zeros(x,y,y_err,Masses)
    if(sim == "300"):
        mask1 = Masses < 13.35
        mask2 = Masses < 14.0
        labels = ["<13.35 bin","13.35 - 14.0 bin",">14.0 bin"]
    elif(sim == "100"):
        mask1 = Masses < 12.3
        mask2 = Masses < 13.0
        labels = ["<12.3 bin","12.3 - 13.0 bin",">13.0 bin"]
    x1 = x[mask1]
    y1 = y[mask1]
    y1_err = y_err[mask1]
    x2 = x[~mask1*mask2]
    y2 = y[~mask1*mask2]
    y2_err = y_err[~mask1*mask2]
    x3 = x[~mask2]
    y3 = y[~mask2]
    y3_err = y_err[~mask2]
    RMSE = [np.nanmean(rmse(x1,y1)),np.nanmean(rmse(x2,y2)),np.nanmean(rmse(x3,y3))]
    R2 = [np.nanmean(r2_score(x1,y1)),np.nanmean(r2_score(x2,y2)),np.nanmean(r2_score(x3,y3))]
    RM_err = [np.nanmean(rel_mean_err(x1,y1)),np.nanmean(rel_mean_err(x2,y2)),np.nanmean(rel_mean_err(x3,y3))]
    chi2 = [np.nanmean(chi_squared(x1,y1,np.exp(y1_err))),np.nanmean(chi_squared(x2,y2,np.exp(y2_err))),np.nanmean(chi_squared(x3,y3,np.exp(y3_err)))]
    cor_cof = [stats.pearsonr(x1,y1)[0],stats.pearsonr(x2,y2)[0],stats.pearsonr(x3,y3)[0]]
    latex += "\\textbf{"+label+"} & " + " & ".join(map(lambda r, chi, rmse, cor_cf: f"{fmt5(rmse)} & {fmt5(cor_cf)} & {fmt5(r)} & {fmt5(chi)}",R2,chi2,RMSE,cor_cof))+" \\\\\n\t"
latex += "\\hline\n"
latex += "\t\\end{tabular*}\n"
latex += "\\label{tab:H1_H1_Multi"+sim+"}\n"
latex += "\\end{table*}\n"
latex += "\\end{center}"
# latex += "\\hline\n\t\\textbf{HI} & "
# for row in rows:
#     latex += " & ".join(map(str, row)) + " \\\\\n"
# latex += "\\hline\n\\end{tabular}"

print(latex)

100
100
100
100
100
100
\begin{center}
\begin{table*}[t]
	\centering
	\caption{Table 5 \HI\ vs \HI\,velocity, and dispersion}
	\begin{tabular*}{15.25cm}{|m{1.4cm}|m{0.9cm}|m{0.7cm}|m{0.7cm}|m{0.7cm}|m{0.9cm}|m{0.7cm}|m{0.7cm}|m{0.7cm}|m{0.9cm}|m{0.7cm}|m{0.7cm}|m{0.7cm}|}
	\hline
	\textbf{Channels} & \multicolumn{4}{c|}{\textbf{$<10^{12.3}$}} & \multicolumn{4}{c|}{\textbf{$10^{12.3}-10^{13.0}$}} & \multicolumn{4}{c|}{\textbf{$>10^{13.0}$}} \\
	\cline{2-13} & RMSE & $\rho$ & $R^2$ & $\chi^2$ & RMSE & $\rho$ & $R^2$ & $\chi^2$ & RMSE & $\rho$ & $R^2$ & $\chi^2$ \\
	\hline
	\textbf{H1} & 0.147 & 0.693 & 0.369 & 0.644 & 0.209 & 0.679 &  -0.1 & 0.657 & 0.536 &  -0.3 & -4.34 & 1.236 \\
	\textbf{H1,vel,disp} & 0.097 & 0.863 &  0.74 & 0.692 & 0.203 & 0.811 & -0.24 & 1.781 & 0.301 &  0.54 & -0.78 & 1.239 \\
	\textbf{H1} & 0.017 & 0.893 & 0.792 & 0.744 & 0.015 & 0.727 & 0.486 & 0.629 & 0.024 & 0.299 & -0.41 & 0.716 \\
	\textbf{H1,vel,disp} & 0.013 & 0.937 & 0.869 & 1.037 &  0.01 & 0.894 & 0.793 